[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_05/ATS_ch5_favar/ATS_ch5_favar.ipynb)

# ATS_ch5_favar

FAVAR of Bernanke, Boivin and Eliasz (2005): three factors and the federal funds rate, two-step estimation with slow-moving factors, VAR(13), 1960:1-2001:8 and 1960:1-2007:12, responses of nine series to a 25 bp shock with residual-bootstrap bands.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 5: Bayesian VAR, factor models and nowcasting. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import hashlib

from scipy import linalg, special

SEED = 2026

FRED_MD_URL = 'https://files.stlouisfed.org/files/htdocs/fred-md/monthly/current.csv'

END_M = '2026-07-01'

FRED_SPEC = [('RPI', 'RPI', 5, 1), ('W875RX1', 'W875RX1', 5, 1), ('INDPRO', 'INDPRO', 5, 1), ('IPFPNSS', 'IPFPNSS', 5, 1), ('IPFINAL', 'IPFINAL', 5, 1), ('IPCONGD', 'IPCONGD', 5, 1), ('IPDCONGD', 'IPDCONGD', 5, 1), ('IPNCONGD', 'IPNCONGD', 5, 1), ('IPBUSEQ', 'IPBUSEQ', 5, 1), ('IPMAT', 'IPMAT', 5, 1), ('IPDMAT', 'IPDMAT', 5, 1), ('IPNMAT', 'IPNMAT', 5, 1), ('IPMANSICS', 'IPMANSICS', 5, 1), ('IPB51222S', 'IPB51222S', 5, 1), ('IPFUELS', 'IPFUELS', 5, 1), ('CUMFNS', 'CUMFNS', 2, 1), ('CLF16OV', 'CLF16OV', 5, 2), ('CE16OV', 'CE16OV', 5, 2), ('UNRATE', 'UNRATE', 2, 2), ('UEMPMEAN', 'UEMPMEAN', 2, 2), ('UEMPLT5', 'UEMPLT5', 5, 2), ('UEMP5TO14', 'UEMP5TO14', 5, 2), ('UEMP15OV', 'UEMP15OV', 5, 2), ('UEMP15T26', 'UEMP15T26', 5, 2), ('UEMP27OV', 'UEMP27OV', 5, 2), ('PAYEMS', 'PAYEMS', 5, 2), ('USPRIV', 'USPRIV', 5, 2), ('USGOOD', 'USGOOD', 5, 2), ('CES1021000001', 'CES1021000001', 5, 2), ('USCONS', 'USCONS', 5, 2), ('MANEMP', 'MANEMP', 5, 2), ('DMANEMP', 'DMANEMP', 5, 2), ('NDMANEMP', 'NDMANEMP', 5, 2), ('SRVPRD', 'SRVPRD', 5, 2), ('USTPU', 'USTPU', 5, 2), ('USWTRADE', 'USWTRADE', 5, 2), ('USTRADE', 'USTRADE', 5, 2), ('USFIRE', 'USFIRE', 5, 2), ('USGOVT', 'USGOVT', 5, 2), ('CES0600000007', 'CES0600000007', 1, 2), ('AWOTMAN', 'AWOTMAN', 2, 2), ('AWHMAN', 'AWHMAN', 1, 2), ('CES0600000008', 'CES0600000008', 6, 2), ('CES2000000008', 'CES2000000008', 6, 2), ('CES3000000008', 'CES3000000008', 6, 2), ('HOUST', 'HOUST', 4, 3), ('HOUSTNE', 'HOUSTNE', 4, 3), ('HOUSTMW', 'HOUSTMW', 4, 3), ('HOUSTS', 'HOUSTS', 4, 3), ('HOUSTW', 'HOUSTW', 4, 3), ('PERMIT', 'PERMIT', 4, 3), ('PERMITNE', 'PERMITNE', 4, 3), ('PERMITMW', 'PERMITMW', 4, 3), ('PERMITS', 'PERMITS', 4, 3), ('PERMITW', 'PERMITW', 4, 3), ('DPCERA3M086SBEA', 'DPCERA3M086SBEA', 5, 4), ('CMRMTSPL', 'CMRMTSPLx', 5, 4), ('RRSFS', 'RETAILx', 5, 4), ('ACOGNO', 'ACOGNO', 5, 4), ('ANDENO', 'ANDENOx', 5, 4), ('AMDMUO', 'AMDMUOx', 5, 4), ('BUSINV', 'BUSINVx', 5, 4), ('ISRATIO', 'ISRATIOx', 2, 4), ('UMCSENT', 'UMCSENTx', 2, 4), ('M1SL', 'M1SL', 6, 5), ('M2SL', 'M2SL', 6, 5), ('M2REAL', 'M2REAL', 5, 5), ('BOGMBASE', 'BOGMBASE', 6, 5), ('TOTRESNS', 'TOTRESNS', 6, 5), ('NONBORRES', 'NONBORRES', 7, 5), ('BUSLOANS', 'BUSLOANS', 6, 5), ('REALLN', 'REALLN', 6, 5), ('NONREVSL', 'NONREVSL', 6, 5), ('DTCOLNVHFNM', 'DTCOLNVHFNM', 6, 5), ('DTCTHFNM', 'DTCTHFNM', 6, 5), ('INVEST', 'INVEST', 6, 5), ('FEDFUNDS', 'FEDFUNDS', 2, 6), ('CPF3M', 'CP3Mx', 2, 6), ('TB3MS', 'TB3MS', 2, 6), ('TB6MS', 'TB6MS', 2, 6), ('GS1', 'GS1', 2, 6), ('GS5', 'GS5', 2, 6), ('GS10', 'GS10', 2, 6), ('AAA', 'AAA', 2, 6), ('BAA', 'BAA', 2, 6), ('EXSZUS', 'EXSZUSx', 5, 6), ('EXJPUS', 'EXJPUSx', 5, 6), ('EXUSUK', 'EXUSUKx', 5, 6), ('EXCAUS', 'EXCAUSx', 5, 6), ('WPSFD49207', 'WPSFD49207', 6, 7), ('WPSFD49502', 'WPSFD49502', 6, 7), ('WPSID61', 'WPSID61', 6, 7), ('WPSID62', 'WPSID62', 6, 7), ('WTISPLC', 'OILPRICEx', 6, 7), ('PPICMM', 'PPICMM', 6, 7), ('CPIAUCSL', 'CPIAUCSL', 6, 7), ('CPIAPPSL', 'CPIAPPSL', 6, 7), ('CPITRNSL', 'CPITRNSL', 6, 7), ('CPIMEDSL', 'CPIMEDSL', 6, 7), ('CUSR0000SAC', 'CUSR0000SAC', 6, 7), ('CUSR0000SAD', 'CUSR0000SAD', 6, 7), ('CUSR0000SAS', 'CUSR0000SAS', 6, 7), ('CPIULFSL', 'CPIULFSL', 6, 7), ('CUSR0000SA0L2', 'CUSR0000SA0L2', 6, 7), ('CUSR0000SA0L5', 'CUSR0000SA0L5', 6, 7), ('PCEPI', 'PCEPI', 6, 7), ('DDURRG3M086SBEA', 'DDURRG3M086SBEA', 6, 7), ('DNDGRG3M086SBEA', 'DNDGRG3M086SBEA', 6, 7), ('DSERRG3M086SBEA', 'DSERRG3M086SBEA', 6, 7)]

SPREADS = [('COMPAPFF', 'CPF3M'), ('TB3SMFFM', 'TB3MS'), ('TB6SMFFM', 'TB6MS'), ('T1YFFM', 'GS1'), ('T5YFFM', 'GS5'), ('T10YFFM', 'GS10'), ('AAAFFM', 'AAA'), ('BAAFFM', 'BAA')]

TCODE = {'RPI': 5, 'W875RX1': 5, 'INDPRO': 5, 'IPFPNSS': 5, 'IPFINAL': 5, 'IPCONGD': 5, 'IPDCONGD': 5, 'IPNCONGD': 5, 'IPBUSEQ': 5, 'IPMAT': 5, 'IPDMAT': 5, 'IPNMAT': 5, 'IPMANSICS': 5, 'IPB51222S': 5, 'IPFUELS': 5, 'CUMFNS': 2, 'CLF16OV': 5, 'CE16OV': 5, 'UNRATE': 2, 'UEMPMEAN': 2, 'UEMPLT5': 5, 'UEMP5TO14': 5, 'UEMP15OV': 5, 'UEMP15T26': 5, 'UEMP27OV': 5, 'PAYEMS': 5, 'USPRIV': 5, 'USGOOD': 5, 'CES1021000001': 5, 'USCONS': 5, 'MANEMP': 5, 'DMANEMP': 5, 'NDMANEMP': 5, 'SRVPRD': 5, 'USTPU': 5, 'USWTRADE': 5, 'USTRADE': 5, 'USFIRE': 5, 'USGOVT': 5, 'CES0600000007': 1, 'AWOTMAN': 2, 'AWHMAN': 1, 'CES0600000008': 6, 'CES2000000008': 6, 'CES3000000008': 6, 'HOUST': 4, 'HOUSTNE': 4, 'HOUSTMW': 4, 'HOUSTS': 4, 'HOUSTW': 4, 'PERMIT': 4, 'PERMITNE': 4, 'PERMITMW': 4, 'PERMITS': 4, 'PERMITW': 4, 'DPCERA3M086SBEA': 5, 'CMRMTSPL': 5, 'RRSFS': 5, 'ACOGNO': 5, 'ANDENO': 5, 'AMDMUO': 5, 'BUSINV': 5, 'ISRATIO': 2, 'UMCSENT': 2, 'M1SL': 6, 'M2SL': 6, 'M2REAL': 5, 'BOGMBASE': 6, 'TOTRESNS': 6, 'NONBORRES': 7, 'BUSLOANS': 6, 'REALLN': 6, 'NONREVSL': 6, 'DTCOLNVHFNM': 6, 'DTCTHFNM': 6, 'INVEST': 6, 'FEDFUNDS': 2, 'CPF3M': 2, 'TB3MS': 2, 'TB6MS': 2, 'GS1': 2, 'GS5': 2, 'GS10': 2, 'AAA': 2, 'BAA': 2, 'EXSZUS': 5, 'EXJPUS': 5, 'EXUSUK': 5, 'EXCAUS': 5, 'WPSFD49207': 6, 'WPSFD49502': 6, 'WPSID61': 6, 'WPSID62': 6, 'WTISPLC': 6, 'PPICMM': 6, 'CPIAUCSL': 6, 'CPIAPPSL': 6, 'CPITRNSL': 6, 'CPIMEDSL': 6, 'CUSR0000SAC': 6, 'CUSR0000SAD': 6, 'CUSR0000SAS': 6, 'CPIULFSL': 6, 'CUSR0000SA0L2': 6, 'CUSR0000SA0L5': 6, 'PCEPI': 6, 'DDURRG3M086SBEA': 6, 'DNDGRG3M086SBEA': 6, 'DSERRG3M086SBEA': 6, 'COMPAPFF': 1, 'TB3SMFFM': 1, 'TB6SMFFM': 1, 'T1YFFM': 1, 'T5YFFM': 1, 'T10YFFM': 1, 'AAAFFM': 1, 'BAAFFM': 1}

GROUP = {'RPI': 1, 'W875RX1': 1, 'INDPRO': 1, 'IPFPNSS': 1, 'IPFINAL': 1, 'IPCONGD': 1, 'IPDCONGD': 1, 'IPNCONGD': 1, 'IPBUSEQ': 1, 'IPMAT': 1, 'IPDMAT': 1, 'IPNMAT': 1, 'IPMANSICS': 1, 'IPB51222S': 1, 'IPFUELS': 1, 'CUMFNS': 1, 'CLF16OV': 2, 'CE16OV': 2, 'UNRATE': 2, 'UEMPMEAN': 2, 'UEMPLT5': 2, 'UEMP5TO14': 2, 'UEMP15OV': 2, 'UEMP15T26': 2, 'UEMP27OV': 2, 'PAYEMS': 2, 'USPRIV': 2, 'USGOOD': 2, 'CES1021000001': 2, 'USCONS': 2, 'MANEMP': 2, 'DMANEMP': 2, 'NDMANEMP': 2, 'SRVPRD': 2, 'USTPU': 2, 'USWTRADE': 2, 'USTRADE': 2, 'USFIRE': 2, 'USGOVT': 2, 'CES0600000007': 2, 'AWOTMAN': 2, 'AWHMAN': 2, 'CES0600000008': 2, 'CES2000000008': 2, 'CES3000000008': 2, 'HOUST': 3, 'HOUSTNE': 3, 'HOUSTMW': 3, 'HOUSTS': 3, 'HOUSTW': 3, 'PERMIT': 3, 'PERMITNE': 3, 'PERMITMW': 3, 'PERMITS': 3, 'PERMITW': 3, 'DPCERA3M086SBEA': 4, 'CMRMTSPL': 4, 'RRSFS': 4, 'ACOGNO': 4, 'ANDENO': 4, 'AMDMUO': 4, 'BUSINV': 4, 'ISRATIO': 4, 'UMCSENT': 4, 'M1SL': 5, 'M2SL': 5, 'M2REAL': 5, 'BOGMBASE': 5, 'TOTRESNS': 5, 'NONBORRES': 5, 'BUSLOANS': 5, 'REALLN': 5, 'NONREVSL': 5, 'DTCOLNVHFNM': 5, 'DTCTHFNM': 5, 'INVEST': 5, 'FEDFUNDS': 6, 'CPF3M': 6, 'TB3MS': 6, 'TB6MS': 6, 'GS1': 6, 'GS5': 6, 'GS10': 6, 'AAA': 6, 'BAA': 6, 'EXSZUS': 6, 'EXJPUS': 6, 'EXUSUK': 6, 'EXCAUS': 6, 'WPSFD49207': 7, 'WPSFD49502': 7, 'WPSID61': 7, 'WPSID62': 7, 'WTISPLC': 7, 'PPICMM': 7, 'CPIAUCSL': 7, 'CPIAPPSL': 7, 'CPITRNSL': 7, 'CPIMEDSL': 7, 'CUSR0000SAC': 7, 'CUSR0000SAD': 7, 'CUSR0000SAS': 7, 'CPIULFSL': 7, 'CUSR0000SA0L2': 7, 'CUSR0000SA0L5': 7, 'PCEPI': 7, 'DDURRG3M086SBEA': 7, 'DNDGRG3M086SBEA': 7, 'DSERRG3M086SBEA': 7, 'COMPAPFF': 6, 'TB3SMFFM': 6, 'TB6SMFFM': 6, 'T1YFFM': 6, 'T5YFFM': 6, 'T10YFFM': 6, 'AAAFFM': 6, 'BAAFFM': 6}

GROUPS = {1: 'Output and income', 2: 'Labour market', 3: 'Housing', 4: 'Consumption, orders, inventories', 5: 'Money and credit', 6: 'Interest and exchange rates', 7: 'Prices'}

SMALL = ['USPRIV', 'CPIAUCSL', 'FEDFUNDS']

MEDIUM = ['USPRIV', 'CPIAUCSL', 'FEDFUNDS', 'PPICMM', 'TOTRESNS', 'M2SL', 'W875RX1', 'DPCERA3M086SBEA', 'INDPRO', 'CUMFNS', 'UNRATE', 'HOUST', 'WPSFD49207', 'PCEPI', 'CES0600000008', 'M1SL', 'GS10', 'BAA', 'TB3MS', 'PERMIT']

FAST_GROUPS = (5, 6)

BGR = {'p': 13, 'window': 120, 'start': '1960-01-01', 'train_end': '1969-12-01', 'H': 12, 'eval1': ('1971-01-01', '2003-12-01'), 'eval2': ('2004-01-01', '2026-07-01'), 'irf': ('1961-01-01', '2002-12-01'), 'Hirf': 48}

GLP = {'lam': (0.2, 0.4), 'mu': (1.0, 1.0), 'dio': (1.0, 1.0)}

SW = {'H': (1, 6, 12), 'kmax': 12, 'pmax': 5, 'start': '1970-01-01', 'eval1': ('1970-01-01', '1998-12-01'), 'eval2': ('1999-01-01', '2026-07-01'), 'targets': ('INDPRO', 'PAYEMS', 'CPIAUCSL')}

BBE = {'K': 3, 'p': 13, 'start': '1960-01-01', 'end': '2001-08-01', 'end2': '2007-12-01', 'H': 48, 'B': 300}

RO_SPEC = {'ip': (('sts_inpr_m', 'M.PRD.B-D.SCA.I21.RO'), 2, 'dlog', 'Industrial production'), 'retail': (('sts_trtu_m', 'M.VOL_SLS.G47.SCA.I21.RO'), 2, 'dlog', 'Retail trade volume'), 'constr': (('sts_copr_m', 'M.PRD.F.SCA.I21.RO'), 2, 'dlog', 'Construction output'), 'ip_ea': (('sts_inpr_m', 'M.PRD.B-D.SCA.I21.EA20'), 2, 'dlog', 'Euro-area industrial production'), 'unemp': (('une_rt_m', 'M.SA.TOTAL.PC_ACT.T.RO'), 1, 'diff', 'Unemployment rate'), 'esi': (('ei_bssi_m_r2', 'M.BS-ESI-I.SA.RO'), 0, 'level', 'Economic sentiment indicator'), 'ici': (('ei_bssi_m_r2', 'M.BS-ICI-BAL.SA.RO'), 0, 'level', 'Industry confidence'), 'sci': (('ei_bssi_m_r2', 'M.BS-SCI-BAL.SA.RO'), 0, 'level', 'Services confidence'), 'rci': (('ei_bssi_m_r2', 'M.BS-RCI-BAL.SA.RO'), 0, 'level', 'Retail confidence'), 'bci': (('ei_bssi_m_r2', 'M.BS-CCI-BAL.SA.RO'), 0, 'level', 'Construction confidence'), 'cci': (('ei_bssi_m_r2', 'M.BS-CSMCI-BAL.SA.RO'), 0, 'level', 'Consumer confidence')}

RO_GDP = ('namq_10_gdp', 'Q.CLV20_MEUR.SCA.B1GQ.RO')

RO = {'start': '2003-01-01', 'eval': ('2013-03-01', '2026-06-01'), 'eval_long': ('2010-03-01', '2026-06-01'), 'r': 2, 'covid': ('2020-04-01', '2020-07-01'), 'gdp_lag': 2, 'vintages': ('M1', 'M2', 'M3', 'M+1'), 'K': 6, 'target': '2026-07-01', 'news_old': '2026-08-01', 'news_new': '2026-09-01'}

MM = np.array([1, 2, 3, 2, 1]) / 3.0

BRIDGE = ('ip', 'retail', 'esi')

MIDAS_X = ('ip', 'esi')

FAVAR_SHOW = [('FEDFUNDS', 'Federal funds rate'), ('INDPRO', 'Industrial production'), ('CPIAUCSL', 'CPI'), ('UNRATE', 'Unemployment rate'), ('CUMFNS', 'Capacity utilisation'), ('DPCERA3M086SBEA', 'Real consumption'), ('HOUST', 'Housing starts'), ('M2SL', 'M2'), ('PAYEMS', 'Payroll employment')]

_FILES = {}

def cached(key, fn):
    """Run fn() once; with the environment variable ATS_CACHE, keep a local copy between runs."""
    if key in _FILES:
        return _FILES[key]
    d = os.environ.get('ATS_CACHE')
    path = os.path.join(d, 'ch5_' + hashlib.md5(key.encode()).hexdigest() + '.pkl') if d else None
    if path and os.path.exists(path):
        _FILES[key] = pd.read_pickle(path)
        return _FILES[key]
    x = fn()
    if path:
        os.makedirs(d, exist_ok=True)
        pd.to_pickle(x, path)
    _FILES[key] = x
    return x


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def fred_md_raw():
    """Raw monthly levels of the FRED-MD series distributed by FRED (one public CSV per series), 1959 onward,
    with the FRED-MD spreads (rate minus the federal funds rate)."""
    def get():
        d = pd.concat([read_fred(s[0]) for s in FRED_SPEC], axis=1)
        d.index = pd.to_datetime(d.index)
        return d.resample('MS').mean().loc['1959-01-01':]
    d = cached('fred_md_raw', get).copy()
    # October 2025: the household survey and the CPI were not collected (US federal shutdown); an isolated missing
    # month inside a series is filled by linear interpolation
    gap = d.isna() & d.shift(1).notna() & d.shift(-1).notna()
    d = d.where(~gap, (d.shift(1) + d.shift(-1)) / 2)
    for name, rate in SPREADS:
        d[name] = d[rate] - d['FEDFUNDS']
    return d


def fred_md_official(url=FRED_MD_URL):
    """The official FRED-MD file (McCracken and Ng 2016), when the St. Louis Fed server allows the download:
    returns (data, transformation codes) or None."""
    try:
        t = pd.read_csv(url, storage_options={'User-Agent': 'Mozilla/5.0'})
    except Exception:
        return None
    tc = t.iloc[0, 1:].astype(int)
    d = t.iloc[1:].dropna(how='all')
    d.index = pd.to_datetime(d.iloc[:, 0], format='%m/%d/%Y')
    return d.iloc[:, 1:].astype(float), tc


def transform(x, tc):
    """FRED-MD transformation codes: 1 level, 2 first difference, 3 second difference, 4 log, 5 log difference,
    6 second log difference, 7 first difference of the percent change."""
    x = pd.Series(x, dtype=float)
    if tc in (4, 5, 6):
        x = np.log(x.where(x > 0))
    if tc == 1 or tc == 4:
        return x
    if tc in (2, 5):
        return x.diff()
    if tc in (3, 6):
        return x.diff().diff()
    if tc == 7:
        return (x / x.shift(1) - 1).diff()
    raise ValueError(tc)


def remove_outliers(X):
    """McCracken and Ng (2016): an observation further than 10 interquartile ranges from the median is missing."""
    med, q1, q3 = X.median(), X.quantile(0.25), X.quantile(0.75)
    return X.where((X - med).abs() <= 10 * (q3 - q1))


def fred_md_panel(raw=None, start='1960-01-01', end=END_M):
    """Stationary FRED-MD-style panel: transformed series, outliers removed (missing values left for the EM step)."""
    raw = fred_md_raw() if raw is None else raw
    X = pd.DataFrame({c: transform(raw[c], TCODE[c]) for c in raw.columns if c in TCODE})
    return remove_outliers(X.loc[start:end])


def bvar_levels(raw, names, start, end):
    """BVAR data as in Banbura, Giannone and Reichlin (2010): 100 x log of the series with transformation code 4, 5, 6,
    levels of rates; delta_i = 1 (random-walk prior) for series that FRED-MD differences, 0 otherwise."""
    Y = {c: (100 * np.log(raw[c]) if TCODE[c] in (4, 5, 6) else raw[c]) for c in names}
    delta = np.array([0.0 if TCODE[c] in (1, 4) else 1.0 for c in names])
    return pd.DataFrame(Y)[names].loc[start:end], delta


def large_names(raw, start=BGR['start'], end=END_M):
    """LARGE system: every series of the panel observed over the whole sample, except code 7 (sign changes)."""
    d = raw.loc[start:end]
    return [c for c in raw.columns if c in TCODE and TCODE[c] != 7 and d[c].notna().all()
            and (TCODE[c] not in (4, 5, 6) or (d[c] > 0).all())]


def ro_raw():
    """Romanian monthly indicators and quarterly GDP (Eurostat SDMX, public, no key)."""
    def get():
        m = pd.DataFrame({k: read_eurostat(*v[0]) for k, v in RO_SPEC.items()})
        return m, read_eurostat(*RO_GDP)
    m, g = cached('ro_raw', get)
    return m.copy(), g.copy()


def ro_panel():
    """Monthly panel (growth rates of activity in %, survey balances in levels, change of the unemployment rate) and
    quarterly GDP growth (q/q, %) placed in the third month of each quarter."""
    m, g = ro_raw()
    X = pd.DataFrame(index=m.index)
    for k, (_, lag, tr, _) in RO_SPEC.items():
        x = m[k]
        X[k] = 100 * np.log(x).diff() if tr == 'dlog' else (x.diff() if tr == 'diff' else x)
    X = X.loc[RO['start']:]
    y = (100 * np.log(g).diff()).loc[RO['start']:]
    y.index = y.index + pd.DateOffset(months=2)          # quarter -> its third month
    return X, y.rename('gdp')


def lagmat(Y, p, const=True):
    """Y_t (t = p..T-1) and X_t = [y_{t-1}', ..., y_{t-p}', 1]."""
    Y = np.asarray(Y, float)
    T = len(Y)
    X = np.hstack([Y[p - j:T - j] for j in range(1, p + 1)])
    if const:
        X = np.hstack([X, np.ones((T - p, 1))])
    return Y[p:], X


def ar_var(y, p):
    """Residual variance of a univariate AR(p) with constant (scale of the Minnesota prior)."""
    yy, X = lagmat(np.asarray(y, float)[:, None], p)
    b = np.linalg.lstsq(X, yy, rcond=None)[0]
    e = yy - X @ b
    return float(e.T @ e / (len(e) - X.shape[1]))


def minnesota(n, p, lam, delta, sig2, const_var=1e6):
    """Minnesota moments in natural conjugate form (Kadiyala and Karlsson 1997; Banbura et al. 2010, eq. 2 with
    theta = 1): prior mean b (k x n), diagonal Omega (k), so that Var(A_l[i, j] | Sigma) = lam^2 Sigma_ii / (l^2 sig2_j);
    Sigma ~ IW(Psi, d) with Psi = diag(sig2), d = n + 2 (prior mean of Sigma = diag(sig2))."""
    k = n * p + 1
    b = np.zeros((k, n))
    b[:n, :n] = np.diag(delta)
    omega = np.concatenate([lam ** 2 / (l ** 2 * np.asarray(sig2)) for l in range(1, p + 1)] + [[const_var]])
    return b, omega, np.diag(sig2), n + 2


def soc_dummies(ybar, p, mu):
    """Sum-of-coefficients prior (Doan, Litterman and Sims 1984): n dummy rows y = diag(ybar)/mu, x = [1' (x) y, 0]."""
    n = len(ybar)
    Yd = np.diag(ybar) / mu
    return Yd, np.hstack([np.tile(Yd, (1, p)), np.zeros((n, 1))])


def dio_dummies(ybar, p, phi):
    """Dummy-initial-observation (single-unit-root) prior (Sims 1993): one row y = ybar'/phi, x = [ybar'/phi, ..., 1/phi]."""
    Yd = (np.asarray(ybar) / phi)[None, :]
    return Yd, np.hstack([np.tile(Yd, (1, p)), [[1.0 / phi]]])


def niw_post(Y, X, b, omega, Psi, d):
    """Posterior of the natural conjugate prior: B | Sigma ~ MN(Bbar, Abar^{-1}, Sigma), Sigma ~ IW(S, T + d), with
    Abar = X'X + Omega^{-1}, Bbar = Abar^{-1}(X'Y + Omega^{-1} b), S = Psi + E'E + (Bbar - b)' Omega^{-1} (Bbar - b)."""
    A = X.T @ X + np.diag(1.0 / omega)
    L = np.linalg.cholesky(A)
    B = linalg.cho_solve((L, True), X.T @ Y + b / omega[:, None])
    E = Y - X @ B
    D = B - b
    S = Psi + E.T @ E + D.T @ (D / omega[:, None])
    return dict(B=B, L=L, S=S, nu=len(Y) + d, E=E)


def niw_logml(Y, X, b, omega, Psi, d):
    """Log marginal likelihood of the natural conjugate model (Giannone, Lenza and Primiceri 2015, appendix):
    -nT/2 ln pi + ln G_n((T+d)/2) - ln G_n(d/2) - n/2 ln|Omega| - n/2 ln|X'X + Omega^{-1}| + d/2 ln|Psi| - (T+d)/2 ln|S|."""
    T, n = Y.shape
    po = niw_post(Y, X, b, omega, Psi, d)
    return float(-n * T / 2 * np.log(np.pi) + special.multigammaln((T + d) / 2, n) - special.multigammaln(d / 2, n)
                 - n / 2 * (np.sum(np.log(omega)) + 2 * np.sum(np.log(np.diag(po['L']))))
                 + d / 2 * np.linalg.slogdet(Psi)[1] - (T + d) / 2 * np.linalg.slogdet(po['S'])[1])


def gamma_pars(mode, sd):
    """Shape and scale of a Gamma density with given mode and standard deviation (GLP 2015)."""
    scale = (-mode + np.sqrt(mode ** 2 + 4 * sd ** 2)) / 2
    return mode / scale + 1, scale


def bvar_setup(Yw, p, delta, sig2=None):
    """Regression form of a VAR(p) on the window Yw and the scale sig2 of each variable (AR(1) residual variances,
    as in GLP 2015, unless given)."""
    Yw = np.asarray(Yw, float)
    Y, X = lagmat(Yw, p)
    if sig2 is None:
        sig2 = np.array([ar_var(Yw[:, i], 1) for i in range(Yw.shape[1])])
    return Y, X, sig2, Yw[:p].mean(0)


def glp_logpost(Y, X, p, delta, sig2, ybar, lam, mu=None, phi=None, hyper=True):
    """GLP (2015) objective: ln p(Y | lam, mu, phi) + ln of the Gamma hyperpriors; the dummies enter as prior
    information, so the marginal likelihood is that of (data + dummies) minus that of the dummies alone."""
    n = Y.shape[1]
    b, om, Psi, d = minnesota(n, p, lam, delta, sig2)
    Yd, Xd = [], []
    if mu is not None:
        a, c = soc_dummies(ybar, p, mu)
        Yd.append(a)
        Xd.append(c)
    if phi is not None:
        a, c = dio_dummies(ybar, p, phi)
        Yd.append(a)
        Xd.append(c)
    if Yd:
        Yd, Xd = np.vstack(Yd), np.vstack(Xd)
        lml = niw_logml(np.vstack([Yd, Y]), np.vstack([Xd, X]), b, om, Psi, d) - niw_logml(Yd, Xd, b, om, Psi, d)
    else:
        lml = niw_logml(Y, X, b, om, Psi, d)
    if hyper:
        for v, (mo, sd) in ((lam, GLP['lam']), (mu, GLP['mu']), (phi, GLP['dio'])):
            if v is not None:
                a, s = gamma_pars(mo, sd)
                lml += stats.gamma.logpdf(v, a, scale=s)
    return lml


def glp_optimize(Yw, p, delta, soc=True, dio=True, x0=None):
    """Posterior mode of the hyperparameters (lam, mu, phi) of GLP (2015), optimised on the log scale."""
    Y, X, sig2, ybar = bvar_setup(Yw, p, delta)
    keys = ['lam'] + (['mu'] if soc else []) + (['phi'] if dio else [])

    def f(z):
        v = dict(zip(keys, np.exp(z)))
        try:
            return -glp_logpost(Y, X, p, delta, sig2, ybar, v['lam'], v.get('mu'), v.get('phi'))
        except np.linalg.LinAlgError:
            return 1e10
    z0 = np.log(x0) if x0 is not None else np.log([0.2] + [1.0] * (len(keys) - 1))
    if len(keys) == 1:
        r = optimize.minimize_scalar(lambda z: f([z]), bounds=(np.log(1e-3), np.log(10)), method='bounded')
        return dict(lam=float(np.exp(r.x)), logpost=float(-r.fun))
    simplex = np.vstack([z0, z0 + 0.7 * np.eye(len(keys))])
    r = optimize.minimize(f, z0, method='Nelder-Mead',
                          options=dict(xatol=1e-3, fatol=1e-4, maxiter=600, initial_simplex=simplex))
    out = dict(zip(keys, map(float, np.exp(r.x))))
    out['logpost'] = float(-r.fun)
    return out


def bvar_fit(Yw, p, delta, lam, soc_mu=None, dio_phi=None, sig2=None, ybar=None, soc_scale=None):
    """Posterior of a BVAR on the window Yw. lam = np.inf gives OLS. soc_mu: sum-of-coefficients tightness, with
    the dummy levels ybar (mean of the first p observations by default; BGR use delta_i x the window mean, soc_scale)."""
    Yw = np.asarray(Yw, float)
    Y, X = lagmat(Yw, p)
    n = Y.shape[1]
    if np.isinf(lam):
        B = np.linalg.lstsq(X, Y, rcond=None)[0]
        E = Y - X @ B
        L = np.linalg.cholesky(X.T @ X) if X.shape[1] < len(Y) else None
        return dict(B=B, S=E.T @ E, nu=len(Y) - X.shape[1], E=E, L=L, p=p)
    if sig2 is None:
        sig2 = np.array([ar_var(Yw[:, i], 1) for i in range(n)])
    b, om, Psi, d = minnesota(n, p, lam, delta, sig2)
    Ys, Xs = [], []
    if soc_mu is not None:
        lev = soc_scale if soc_scale is not None else Yw[:p].mean(0)
        a, c = soc_dummies(lev, p, soc_mu)
        Ys.append(a)
        Xs.append(c)
    if dio_phi is not None:
        a, c = dio_dummies(Yw[:p].mean(0) if ybar is None else ybar, p, dio_phi)
        Ys.append(a)
        Xs.append(c)
    if Ys:
        Y, X = np.vstack(Ys + [Y]), np.vstack(Xs + [X])
    po = niw_post(Y, X, b, om, Psi, d)
    po['p'] = p
    return po


def companion_roots(B, n, p):
    A = B[:n * p].T
    F = np.zeros((n * p, n * p))
    F[:n] = A
    F[n:, :-n] = np.eye(n * (p - 1))
    return np.abs(np.linalg.eigvals(F))


def var_forecast(B, Yhist, p, H):
    """Iterated point forecasts y_{T+1..T+H} with coefficients B = [A_1; ...; A_p; c]."""
    hist = list(np.asarray(Yhist, float)[-p:])
    out = []
    for _ in range(H):
        x = np.concatenate([hist[-j] for j in range(1, p + 1)] + [[1.0]])
        yh = x @ B
        out.append(yh)
        hist.append(yh)
    return np.array(out)


def niw_draw(po, rng):
    """One draw (B, Sigma) from the Normal-inverse-Wishart posterior."""
    Sig = stats.invwishart.rvs(df=po['nu'], scale=po['S'], random_state=rng)
    Sig = np.atleast_2d(Sig)
    Z = rng.standard_normal(po['B'].shape)
    B = po['B'] + linalg.solve_triangular(po['L'].T, Z, lower=False) @ np.linalg.cholesky(Sig).T
    return B, Sig


def irf_shock(B, b0, n, p, H):
    """Responses theta_h = sum_l A_l theta_{h-l} to an impact vector b0, h = 0..H."""
    A = [B[(l - 1) * n:l * n].T for l in range(1, p + 1)]
    th = [np.asarray(b0, float)]
    for h in range(1, H + 1):
        th.append(sum(A[l - 1] @ th[h - l] for l in range(1, min(h, p) + 1)))
    return np.array(th)


def recursive_impact(Sig, k):
    """Impact column of the shock to variable k under a Cholesky ordering, scaled to a unit impact on variable k."""
    P = np.linalg.cholesky(Sig)
    return P[:, k] / P[k, k]


def band_plot(ax, h, mid, lo, hi, color, label, lo2=None, hi2=None):
    """Point estimate with a shaded band (and an optional inner band)."""
    ax.fill_between(h, lo, hi, color=color, alpha=0.15, lw=0, label='_band')
    if lo2 is not None:
        ax.fill_between(h, lo2, hi2, color=color, alpha=0.28, lw=0, label='_band2')
    ax.plot(h, mid, color=color, lw=1.8, label=label)
    ax.axhline(0, color=st.DarkText, lw=0.6)


def plain_log_axis(ax, subs=(1.0, 2.0, 5.0)):
    """Log scale with plain decimal tick labels."""
    from matplotlib.ticker import FuncFormatter, LogLocator
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=subs))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ''))


def patch(color, alpha=0.2):
    from matplotlib.patches import Patch
    return Patch(color=color, alpha=alpha)


def recessions(ax, rec, ymin=0, ymax=1):
    """Shade NBER recessions (FRED USREC) on a date axis."""
    r = rec.astype(int)
    starts = r.index[(r.diff() == 1)]
    ends = r.index[(r.diff() == -1)]
    if len(r) and r.iloc[0] == 1:
        starts = starts.insert(0, r.index[0])
    for s, e in zip(starts, list(ends) + [r.index[-1]] * (len(starts) - len(ends))):
        ax.axvspan(s, e, color=st.Amber, alpha=0.18, lw=0, label='_rec')


def favar_data(end):
    """BBE (2005) panel: balanced transformed series 1960:1-end, standardised; slow-moving series; the funds rate."""
    raw = fred_md_raw()
    # prices, wages and money enter as monthly growth rates (code 6 -> 5), so that level responses cumulate once
    X = pd.DataFrame({c: transform(raw[c], 5 if TCODE[c] == 6 else TCODE[c]) for c in raw.columns if c in TCODE})
    X = X.loc[BBE['start']:end]
    X = X.loc[:, X.notna().all()]
    X = X.drop(columns=[c for c in X.columns if c == 'FEDFUNDS'])
    Z = (X - X.mean()) / X.std()
    slow = [c for c in Z.columns if GROUP[c] not in FAST_GROUPS]
    return Z, slow, raw['FEDFUNDS'].loc[BBE['start']:end], X.std()


def favar_estimate(Z, slow, ffr, K):
    """Two-step FAVAR of BBE (2005, Section III): C = first K+1... principal components of all series; slow factors
    from the slow-moving series; F = C - b_Y Y from the regression of C on the slow factors and the funds rate."""
    def pcs(M, k):
        U, s, Vt = np.linalg.svd(M, full_matrices=False)
        return np.sqrt(len(M)) * U[:, :k]
    C = pcs(Z.values, K)
    Fs = pcs(Z[slow].values, K)
    y = ffr.values
    R = np.column_stack([np.ones(len(y)), Fs, y])
    b = np.linalg.lstsq(R, C, rcond=None)[0]
    F = C - np.outer(y, b[-1])
    S = np.column_stack([F, y])
    Lam = np.linalg.lstsq(np.column_stack([np.ones(len(y)), S]), Z.values, rcond=None)[0][1:]
    return S, Lam


def ols_var(S, p):
    Y, X = lagmat(S, p)
    B = np.linalg.lstsq(X, Y, rcond=None)[0]
    U = Y - X @ B
    return B, U, U.T @ U / (len(U) - X.shape[1])


def favar_irf_obs(B, Sig, Lam, sd, cols, K, p, H):
    """Responses of the observed series to a 25 bp funds-rate shock: loadings times the factor responses, rescaled to
    the original units and cumulated for differenced series."""
    k = K
    imp = recursive_impact(Sig, k) * 0.25
    th = irf_shock(B, imp, K + 1, p, H)
    out = {}
    for c, _ in FAVAR_SHOW:
        if c == 'FEDFUNDS':
            out[c] = th[:, k]
            continue
        j = cols.index(c)
        x = th @ Lam[:, j] * sd[c]
        tc = 5 if TCODE[c] == 6 else TCODE[c]
        if tc in (2, 5):
            x = np.cumsum(x)
        elif tc in (3, 6):
            x = np.cumsum(np.cumsum(x))
        out[c] = x * (100 if tc in (4, 5, 6) else 1)
    return out


def favar_run(end, K, B=BBE['B'], seed=SEED):
    """Point responses and residual-bootstrap 90% bands (the factors are treated as data, as in BBE)."""
    Z, slow, ffr, sd = favar_data(end)
    p, H = BBE['p'], BBE['H']
    S, Lam = favar_estimate(Z, slow, ffr, K)
    Bv, U, Sig = ols_var(S, p)
    cols = list(Z.columns)
    pt = favar_irf_obs(Bv, Sig, Lam, sd, cols, K, p, H)
    rng = np.random.default_rng(seed)
    Yl, X = lagmat(S, p)
    draws = {c: [] for c in pt}
    for _ in range(B):
        idx = rng.integers(0, len(U), len(U))
        Sb = np.zeros_like(S)
        Sb[:p] = S[:p]
        for t in range(p, len(S)):
            x = np.concatenate([Sb[t - j] for j in range(1, p + 1)] + [[1.0]])
            Sb[t] = x @ Bv + U[idx[t - p]]
        Bb, Ub, Sb_ = ols_var(Sb, p)
        r = favar_irf_obs(Bb, Sb_, Lam, sd, cols, K, p, H)
        for c in r:
            draws[c].append(r[c])
    bands = {c: np.quantile(np.array(v), [0.05, 0.95], axis=0) for c, v in draws.items()} if B else None
    return pt, bands, dict(T=int(len(S)), N=int(Z.shape[1]), Nslow=len(slow))


def fig_favar(save_it=True, B=BBE['B']):
    """BBE (2005): K = 3 factors and the funds rate, VAR(13), 1960:1-2001:8; responses to a 25 bp shock with 90%
    bootstrap bands; dashed: the sample extended to 2007:12."""
    pt, bd, info = favar_run(BBE['end'], BBE['K'], B=B)
    pt2, _, info2 = favar_run(BBE['end2'], BBE['K'], B=0)
    H = BBE['H']
    hh = np.arange(H + 1)
    fig, axs = plt.subplots(3, 3, figsize=(12, 7.2), sharex=True)
    out = dict(info=info, info2=info2)
    for ax, (c, lab), col in zip(axs.ravel(), FAVAR_SHOW, st.PALETTE * 2):
        band_plot(ax, hh, pt[c], bd[c][0], bd[c][1], st.MainBlue, 'FAVAR, 1960-2001')
        ax.plot(hh, pt2[c], color=st.IDAred, lw=1.4, ls='--', label='FAVAR, 1960-2007')
        ax.set_title(lab, fontsize=12)
        out[c] = dict(h12=float(pt[c][12]), h48=float(pt[c][48]), lo48=float(bd[c][0][48]), hi48=float(bd[c][1][48]),
                      min=float(pt[c].min()), argmin=int(pt[c].argmin()), max=float(pt[c].max()),
                      argmax=int(pt[c].argmax()), h48_2=float(pt2[c][48]), max_2=float(pt2[c][:24].max()))
    for ax in axs[-1]:
        ax.set_xlabel('months')
    st.fig_legend_bottom(fig, [axs[0, 0].lines[0], patch(st.MainBlue, 0.15), axs[0, 0].lines[-1]],
                         ['FAVAR, 1960:1-2001:8', '90% bootstrap band', 'FAVAR, 1960:1-2007:12'], ncol=3, y=0.0)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    save('ats_ch5_favar', save_it)
    return out

## Run

In [ ]:
print(fig_favar(B=100)["INDPRO"])

![ats_ch5_favar](./ats_ch5_favar.png)

Output: `ats_ch5_favar.pdf`